# LeetCode 506 — Relative Ranks bằng Python

**Link đề:** https://leetcode.com/problems/relative-ranks/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Heap (Priority Queue)

```python
class Solution:
    def findRelativeRanks(self, score: list[int]) -> list[str]:
        ...
```

## Mục tiêu học
1. Phân biệt **vị trí trong mảng gốc** (vận động viên thứ `i`) và **thứ hạng** (đứng thứ mấy).
2. Học `sorted` với `key=` và `reverse=True`, `lambda`, sắp xếp **chỉ số** theo giá trị (argsort).
3. Đi từ cách đếm "bao nhiêu người hơn mình" O(n²) → sắp xếp O(n log n) → heap.
4. Kỹ thuật "**ghi kết quả về đúng vị trí gốc**" sau khi sắp xếp.
5. Trace, test, so sánh 4 cách, luyện biến thể (có điểm hoà, top-k).

## 1. Đề bài

Cho mảng `score` độ dài `n`, `score[i]` là điểm của vận động viên thứ `i`. Mọi điểm **khác nhau**.

Người điểm cao nhất xếp hạng 1, cao nhì hạng 2, ... Danh hiệu:
- Hạng 1 → `"Gold Medal"`
- Hạng 2 → `"Silver Medal"`
- Hạng 3 → `"Bronze Medal"`
- Hạng `x ≥ 4` → chuỗi `"x"` (ví dụ `"4"`).

Trả về `answer` cùng độ dài, `answer[i]` là danh hiệu của vận động viên thứ `i` (**giữ nguyên thứ tự gốc**).

| score | Output |
|---|---|
| `[5,4,3,2,1]` | `["Gold Medal","Silver Medal","Bronze Medal","4","5"]` |
| `[10,3,8,9,4]` | `["Gold Medal","5","Bronze Medal","Silver Medal","4"]` |

**Constraints:** `1 <= n <= 10^4`; `0 <= score[i] <= 10^6`; mọi giá trị khác nhau.

### Phân tích bằng tay ví dụ 2

| i | score[i] | Số người điểm cao hơn | Hạng | Danh hiệu |
|---:|---:|---:|---:|---|
| 0 | 10 | 0 | 1 | Gold Medal |
| 1 | 3 | 4 | 5 | "5" |
| 2 | 8 | 2 (10, 9) | 3 | Bronze Medal |
| 3 | 9 | 1 (10) | 2 | Silver Medal |
| 4 | 4 | 3 (10, 9, 8) | 4 | "4" |

Nhận xét: **hạng = 1 + số người điểm cao hơn**. Đây chính là ý tưởng cho cách trực giác.

## 2. Nền tảng Python

### 2.1. Hàm đổi hạng → danh hiệu

Tách riêng phần "định dạng" để code chính gọn. Dùng list `MEDALS` cho 3 hạng đầu và `str(rank)` cho phần còn lại.

In [1]:
MEDALS = ["Gold Medal", "Silver Medal", "Bronze Medal"]

def rank_to_label(rank):          # rank bắt đầu từ 1
    return MEDALS[rank - 1] if rank <= 3 else str(rank)

print([rank_to_label(r) for r in range(1, 7)])

['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5', '6']


### 2.2. `sorted(...)`, `reverse=True`, `key=` và `lambda`

- `sorted(a)` trả về **list mới** đã sắp tăng dần; `a` không đổi. (`a.sort()` sắp **tại chỗ**, trả `None`.)
- `reverse=True` → giảm dần.
- `key=f` → sắp theo `f(x)` thay vì `x`.
- `lambda i: score[i]` là **hàm ẩn danh** một dòng, tương đương `def f(i): return score[i]`.

In [2]:
score = [10, 3, 8, 9, 4]
print(sorted(score))                    # tăng dần
print(sorted(score, reverse=True))      # giảm dần
print(score)                            # gốc không đổi
words = ["bb", "a", "ccc"]
print(sorted(words, key=len))           # theo độ dài
f = lambda x: -x
print(f(5), sorted(score, key=lambda x: -x))

[3, 4, 8, 9, 10]
[10, 9, 8, 4, 3]
[10, 3, 8, 9, 4]
['a', 'bb', 'ccc']
-5 [10, 9, 8, 4, 3]


### 2.3. Sắp xếp **chỉ số** theo giá trị (argsort)

Ta không chỉ muốn biết điểm theo thứ tự, mà còn muốn biết **ai** (chỉ số `i`) đứng ở thứ hạng đó:

```python
order = sorted(range(n), key=lambda i: score[i], reverse=True)
```
`order[0]` là chỉ số người hạng 1, `order[1]` là người hạng 2, ...

In [3]:
score = [10, 3, 8, 9, 4]
order = sorted(range(len(score)), key=lambda i: score[i], reverse=True)
print("order =", order)
for place, i in enumerate(order, start=1):
    print(f"hạng {place}: vận động viên {i} (điểm {score[i]})")

order = [0, 3, 2, 4, 1]
hạng 1: vận động viên 0 (điểm 10)
hạng 2: vận động viên 3 (điểm 9)
hạng 3: vận động viên 2 (điểm 8)
hạng 4: vận động viên 4 (điểm 4)
hạng 5: vận động viên 1 (điểm 3)


### 2.4. Tạo list rỗng kích thước n và ghi vào vị trí bất kỳ

`ans = [""] * n` tạo list `n` chuỗi rỗng; sau đó `ans[i] = ...` ghi vào đúng ô `i`. (Với phần tử bất biến như chuỗi/số, nhân list như vậy là an toàn.)

In [4]:
ans = [""] * 5
ans[3] = "Silver Medal"
print(ans)

['', '', '', 'Silver Medal', '']


## 3. Cách 1 — Trực giác: đếm số người hơn mình

`rank(i) = 1 + số j có score[j] > score[i]`. Hai vòng lặp lồng nhau.

In [5]:
def relative_ranks_count(score):
    n = len(score)
    ans = []
    for i in range(n):
        higher = 0
        for j in range(n):
            if score[j] > score[i]:
                higher += 1
        ans.append(rank_to_label(higher + 1))
    return ans

print(relative_ranks_count([5, 4, 3, 2, 1]))
print(relative_ranks_count([10, 3, 8, 9, 4]))

['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5']
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


### Nút thắt

`n ≤ 10^4` → `n² = 10^8` phép so sánh: **quá chậm** với Python (cỡ chục giây). Ta đo số so sánh:

In [6]:
for n in [10, 100, 1000, 10000]:
    print(f"n={n:6d}: số so sánh = {n * n:,}")

n=    10: số so sánh = 100
n=   100: số so sánh = 10,000
n=  1000: số so sánh = 1,000,000
n= 10000: số so sánh = 100,000,000


Mỗi người đều tự "so với tất cả" → lặp lại rất nhiều. Nếu **sắp xếp một lần**, thứ hạng của mọi người đọc ra ngay từ vị trí trong danh sách đã sắp.

## 4. Insight — sắp xếp một lần, rồi trả kết quả về vị trí gốc

1. Sắp các **chỉ số** theo điểm giảm dần → `order`.
2. Người ở vị trí `place` (0-based) trong `order` có hạng `place + 1`.
3. Ghi `ans[order[place]] = rank_to_label(place + 1)` — tức là **ghi về đúng ô của người đó trong mảng gốc**.

## 5. Lời giải chính — argsort

In [7]:
def relative_ranks_argsort(score):
    n = len(score)
    order = sorted(range(n), key=lambda i: score[i], reverse=True)
    ans = [""] * n
    for place, i in enumerate(order):
        ans[i] = rank_to_label(place + 1)
    return ans

print(relative_ranks_argsort([5, 4, 3, 2, 1]))
print(relative_ranks_argsort([10, 3, 8, 9, 4]))

['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5']
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


## 6. TRACE từng bước

`score = [10, 3, 8, 9, 4]` → `order = [0, 3, 2, 4, 1]`.

| place | i = order[place] | score[i] | hạng | ghi vào ans[i] | ans sau |
|---:|---:|---:|---:|---|---|
| 0 | 0 | 10 | 1 | Gold Medal | `[G, _, _, _, _]` |
| 1 | 3 | 9 | 2 | Silver Medal | `[G, _, _, S, _]` |
| 2 | 2 | 8 | 3 | Bronze Medal | `[G, _, B, S, _]` |
| 3 | 4 | 4 | 4 | "4" | `[G, _, B, S, 4]` |
| 4 | 1 | 3 | 5 | "5" | `[G, 5, B, S, 4]` |

In [8]:
def trace_relative_ranks(score):
    n = len(score)
    order = sorted(range(n), key=lambda i: score[i], reverse=True)
    print("score =", score)
    print("order =", order, "(chỉ số theo điểm giảm dần)")
    ans = [""] * n
    for place, i in enumerate(order):
        label = rank_to_label(place + 1)
        ans[i] = label
        short = [a[:1] if a and not a[0].isdigit() else (a or "_") for a in ans]
        print(f"place={place} -> i={i}, score={score[i]:>3}, hạng={place + 1} -> ans[{i}]='{label}' | {short}")
    return ans

print(trace_relative_ranks([10, 3, 8, 9, 4]))

score = [10, 3, 8, 9, 4]
order = [0, 3, 2, 4, 1] (chỉ số theo điểm giảm dần)
place=0 -> i=0, score= 10, hạng=1 -> ans[0]='Gold Medal' | ['G', '_', '_', '_', '_']
place=1 -> i=3, score=  9, hạng=2 -> ans[3]='Silver Medal' | ['G', '_', '_', 'S', '_']
place=2 -> i=2, score=  8, hạng=3 -> ans[2]='Bronze Medal' | ['G', '_', 'B', 'S', '_']
place=3 -> i=4, score=  4, hạng=4 -> ans[4]='4' | ['G', '_', 'B', 'S', '4']
place=4 -> i=1, score=  3, hạng=5 -> ans[1]='5' | ['G', '5', 'B', 'S', '4']
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


## 7. Các cách khác

### 7.1. Sắp điểm + `dict` điểm → hạng
Vì điểm **khác nhau**, ta có thể map trực tiếp `điểm → hạng`, rồi duyệt `score` theo thứ tự gốc.

In [9]:
def relative_ranks_dict(score):
    rank_of = {s: place + 1 for place, s in enumerate(sorted(score, reverse=True))}
    print("rank_of =", rank_of)
    return [rank_to_label(rank_of[s]) for s in score]

print(relative_ranks_dict([10, 3, 8, 9, 4]))

rank_of = {10: 1, 9: 2, 8: 3, 4: 4, 3: 5}
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


### 7.2. Heap (hàng đợi ưu tiên) — `heapq`

`heapq` là **min-heap**: `heappop` luôn lấy phần tử **nhỏ nhất** trong `O(log n)`. Muốn lấy lớn nhất trước, ta đẩy `(-score, i)`.
- `heapq.heapify(list)` biến list thành heap trong `O(n)`.
- Tuple được so sánh theo phần tử đầu trước.

In [10]:
import heapq
h = [5, 1, 4]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), heapq.heappop(h))   # 1 4 5

def relative_ranks_heap(score):
    heap = [(-s, i) for i, s in enumerate(score)]
    heapq.heapify(heap)
    ans = [""] * len(score)
    place = 1
    while heap:
        _, i = heapq.heappop(heap)
        ans[i] = rank_to_label(place)
        place += 1
    return ans

print(relative_ranks_heap([10, 3, 8, 9, 4]))

1 4 5
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


### 7.3. Counting (mảng đánh dấu theo điểm) — khi điểm nhỏ
Điểm ≤ `10^6`: tạo mảng `pos` dài `max(score)+1`, `pos[s] = i`. Duyệt điểm từ cao xuống thấp. Thời gian `O(n + M)` với `M = max(score)`; chỉ đáng dùng khi `M` không quá lớn so với `n`.

In [11]:
def relative_ranks_counting(score):
    M = max(score)
    pos = [-1] * (M + 1)
    for i, s in enumerate(score):
        pos[s] = i
    ans = [""] * len(score)
    place = 1
    for s in range(M, -1, -1):        # từ điểm cao xuống thấp
        if pos[s] != -1:
            ans[pos[s]] = rank_to_label(place)
            place += 1
    return ans

print(relative_ranks_counting([10, 3, 8, 9, 4]))

['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


## 8. Code nộp LeetCode

### Bản argsort (khuyên dùng)

In [12]:
class Solution:
    def findRelativeRanks(self, score: list[int]) -> list[str]:
        medals = ["Gold Medal", "Silver Medal", "Bronze Medal"]
        n = len(score)
        order = sorted(range(n), key=lambda i: score[i], reverse=True)
        ans = [""] * n
        for place, i in enumerate(order):
            ans[i] = medals[place] if place < 3 else str(place + 1)
        return ans

print(Solution().findRelativeRanks([5, 4, 3, 2, 1]))
print(Solution().findRelativeRanks([10, 3, 8, 9, 4]))

['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5']
['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


### Bản heap

In [13]:
class SolutionHeap:
    def findRelativeRanks(self, score: list[int]) -> list[str]:
        medals = ["Gold Medal", "Silver Medal", "Bronze Medal"]
        heap = [(-s, i) for i, s in enumerate(score)]
        heapq.heapify(heap)
        ans = [""] * len(score)
        place = 0
        while heap:
            _, i = heapq.heappop(heap)
            ans[i] = medals[place] if place < 3 else str(place + 1)
            place += 1
        return ans

print(SolutionHeap().findRelativeRanks([10, 3, 8, 9, 4]))

['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


(Khi nộp, đổi tên lớp thành `Solution`; nhớ `import heapq` ở đầu file — LeetCode thường đã import sẵn.)

## 9. Vì sao đúng?

- Sau khi sắp giảm dần, phần tử ở vị trí `place` (0-based) có đúng `place` người điểm cao hơn (vì điểm khác nhau) → hạng `place + 1`. Khớp định nghĩa "hạng = 1 + số người hơn mình".
- `order` là một **hoán vị** của `0..n-1` → mỗi ô `ans[i]` được ghi **đúng một lần**, không ô nào bị bỏ trống.
- Ghi vào `ans[i]` (chỉ số gốc) chứ không phải `ans[place]` → giữ đúng thứ tự vận động viên.

## 10. Độ phức tạp

| Cách | Time | Space phụ | Ghi chú |
|---|---:|---:|---|
| Đếm người hơn mình | `O(n²)` | `O(1)` | quá chậm với n = 10^4 |
| Argsort | `O(n log n)` | `O(n)` | chuẩn |
| Sort + dict | `O(n log n)` | `O(n)` | cần điểm khác nhau |
| Heap | `O(n log n)` (heapify `O(n)` + n lần pop `O(log n)`) | `O(n)` | hữu ích khi chỉ cần top-k |
| Counting theo điểm | `O(n + M)` | `O(M)` | `M = max(score)` ≤ 10^6 |

Đo thời gian trên `n = 10^4` để cảm nhận (chỉ minh hoạ, số đo tuỳ máy):

In [14]:
import random, time
random.seed(506)
big = random.sample(range(10**6 + 1), 10**4)
for f in [relative_ranks_argsort, relative_ranks_heap, relative_ranks_counting]:
    t0 = time.perf_counter(); f(big); t1 = time.perf_counter()
    print(f"{f.__name__:26s}: {t1 - t0:.4f} s")
small = big[:1500]
t0 = time.perf_counter(); relative_ranks_count(small); t1 = time.perf_counter()
print(f"relative_ranks_count (chỉ n=1500): {t1 - t0:.4f} s  -> n=10^4 sẽ chậm hơn ~44 lần")

relative_ranks_argsort    : 0.0035 s
relative_ranks_heap       : 0.0070 s
relative_ranks_counting   : 0.0321 s


relative_ranks_count (chỉ n=1500): 0.0972 s  -> n=10^4 sẽ chậm hơn ~44 lần


## 11. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1", s.findRelativeRanks([5, 4, 3, 2, 1]), ["Gold Medal", "Silver Medal", "Bronze Medal", "4", "5"])
check("đề 2", s.findRelativeRanks([10, 3, 8, 9, 4]), ["Gold Medal", "5", "Bronze Medal", "Silver Medal", "4"])
check("1 người", s.findRelativeRanks([42]), ["Gold Medal"])
check("2 người", s.findRelativeRanks([1, 2]), ["Silver Medal", "Gold Medal"])
check("3 người tăng dần", s.findRelativeRanks([0, 1, 2]), ["Bronze Medal", "Silver Medal", "Gold Medal"])
check("hạng 2 chữ số", s.findRelativeRanks(list(range(11)))[:2], ["11", "10"])

PASS đề 1: ['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5']
PASS đề 2: ['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']
PASS 1 người: ['Gold Medal']
PASS 2 người: ['Silver Medal', 'Gold Medal']
PASS 3 người tăng dần: ['Bronze Medal', 'Silver Medal', 'Gold Medal']
PASS hạng 2 chữ số: ['11', '10']


In [16]:
random.seed(5060)
for _ in range(300):
    n = random.randint(1, 40)
    sc = random.sample(range(0, 200), n)
    r = relative_ranks_count(sc)
    assert r == relative_ranks_argsort(sc) == relative_ranks_heap(sc) == relative_ranks_counting(sc)
    assert r == Solution().findRelativeRanks(sc) == SolutionHeap().findRelativeRanks(sc)
    rank_of = {v: p + 1 for p, v in enumerate(sorted(sc, reverse=True))}   # logic của cách dict (bản không in)
    assert r == [rank_to_label(rank_of[v]) for v in sc]
print("300 bộ ngẫu nhiên: các cách khớp với brute force — PASS")

300 bộ ngẫu nhiên: các cách khớp với brute force — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: trả kết quả theo thứ tự đã sắp
```python
return [rank_to_label(p + 1) for p in range(n)]   # luôn là Gold, Silver, Bronze, 4, ...
```
Đề cần kết quả theo **thứ tự vận động viên gốc**.

### Lỗi 2: sắp tăng dần (quên `reverse=True`)
Người điểm thấp nhất nhận Gold Medal.

### Lỗi 3: dùng `score.index(s)` trong vòng lặp
Đúng nhưng `O(n)` mỗi lần → tổng `O(n²)`, mất lợi ích của sắp xếp.

### Lỗi 4: hạng lệch 1 (off-by-one)
Dùng `str(place)` thay vì `str(place + 1)` → người thứ 4 thành `"3"`.

In [17]:
def buggy_sorted_order(score):
    return [rank_to_label(p + 1) for p in range(len(score))]

def buggy_ascending(score):
    order = sorted(range(len(score)), key=lambda i: score[i])
    ans = [""] * len(score)
    for place, i in enumerate(order):
        ans[i] = rank_to_label(place + 1)
    return ans

def buggy_off_by_one(score):
    order = sorted(range(len(score)), key=lambda i: score[i], reverse=True)
    ans = [""] * len(score)
    for place, i in enumerate(order):
        ans[i] = ["Gold Medal", "Silver Medal", "Bronze Medal"][place] if place < 3 else str(place)
    return ans

ex = [10, 3, 8, 9, 4]
print("thứ tự sai   :", buggy_sorted_order(ex))
print("tăng dần     :", buggy_ascending(ex))
print("lệch 1       :", buggy_off_by_one(ex))
print("đúng         :", Solution().findRelativeRanks(ex))

thứ tự sai   : ['Gold Medal', 'Silver Medal', 'Bronze Medal', '4', '5']
tăng dần     : ['5', 'Gold Medal', 'Bronze Medal', '4', 'Silver Medal']
lệch 1       : ['Gold Medal', '4', 'Bronze Medal', 'Silver Medal', '3']
đúng         : ['Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '4']


## 13. Mẹo nhận dạng pattern

- "Thứ hạng / thứ tự của mỗi phần tử nhưng **trả theo vị trí gốc**" → **argsort**: `sorted(range(n), key=lambda i: a[i])` rồi ghi `ans[i]`.
- Giá trị khác nhau → có thể map `giá trị → hạng` bằng dict (coordinate compression — **nén toạ độ**).
- Chỉ cần **top-k** → heap (`heapq.nlargest(k, ...)`) tốt hơn sắp xếp toàn bộ.
- Giá trị nguyên nhỏ, miền hẹp → counting/bucket.

## 14. Biến thể

### 14.1. Rank Transform of an Array (LeetCode 1331) — có giá trị **trùng**
Giá trị bằng nhau nhận **cùng hạng**, hạng nhỏ nhất là 1, hạng liên tiếp không nhảy cóc. Dùng `sorted(set(arr))`.

In [18]:
def rank_transform(arr):
    rank_of = {v: r for r, v in enumerate(sorted(set(arr)), start=1)}
    return [rank_of[v] for v in arr]

print(rank_transform([40, 10, 20, 30]))     # [4, 1, 2, 3]
print(rank_transform([100, 100, 100]))      # [1, 1, 1]
print(rank_transform([37, 12, 28, 9, 100, 56, 80, 5, 12]))

[4, 1, 2, 3]
[1, 1, 1]
[5, 3, 4, 2, 8, 6, 7, 1, 3]


### 14.2. Chỉ lấy 3 người đạt huy chương — `heapq.nlargest`

In [19]:
score = [10, 3, 8, 9, 4]
top3 = heapq.nlargest(3, range(len(score)), key=lambda i: score[i])
print("chỉ số 3 người đứng đầu:", top3)

chỉ số 3 người đứng đầu: [0, 3, 2]


## 15. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Với `score = [7, 30, 2, 15, 22, 1]`, viết `order` và `answer`.

### Bài 2 — Điền TODO (argsort)

In [20]:
def exercise_relative_ranks(score):
    n = len(score)
    # TODO 1: order = các chỉ số sắp theo điểm giảm dần
    ans = [""] * n
    # TODO 2: duyệt place, i và ghi ans[i]
    return ans

# assert exercise_relative_ranks([10, 3, 8, 9, 4]) == ["Gold Medal", "5", "Bronze Medal", "Silver Medal", "4"]

### Bài 3 — Hạng dạng số
Trả về list số nguyên `ranks` với `ranks[i]` là hạng của người `i` (1 = điểm cao nhất).

In [21]:
def numeric_ranks(score):
    # TODO
    pass

# assert numeric_ranks([10, 3, 8, 9, 4]) == [1, 5, 3, 2, 4]

### Bài 4 — Có điểm hoà (kiểu thi đấu "1, 2, 2, 4")
Điểm có thể trùng. Người bằng điểm nhận cùng hạng; hạng tiếp theo **nhảy cóc** (standard competition ranking). Trả list số.

In [22]:
def competition_ranks(score):
    # TODO
    pass

# assert competition_ranks([90, 80, 80, 70]) == [1, 2, 2, 4]

### Bài 5 — Tên người đạt huy chương
Cho `names` và `score` cùng độ dài, trả về dict `{"Gold Medal": tên, "Silver Medal": tên, "Bronze Medal": tên}` (chỉ những huy chương có người nhận).

In [23]:
def medal_winners(names, score):
    # TODO
    pass

# assert medal_winners(["An", "Bình", "Chi", "Dũng"], [5, 9, 7, 1]) == {"Gold Medal": "Bình", "Silver Medal": "Chi", "Bronze Medal": "An"}
# assert medal_winners(["An"], [3]) == {"Gold Medal": "An"}

### Bài 6 — Mini challenge: hạng theo nhiều tiêu chí
Mỗi người có `(điểm, thời_gian)`. Xếp hạng theo điểm **giảm dần**, nếu bằng điểm thì thời gian **tăng dần**. Trả list hạng số theo thứ tự gốc (giả sử không có cặp nào trùng hoàn toàn).

In [24]:
def multi_key_ranks(records):
    # TODO
    pass

# assert multi_key_ranks([(90, 30), (95, 50), (90, 20)]) == [3, 1, 2]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sắp điểm giảm dần | ghi chỉ số tương ứng | dùng `trace_relative_ranks` để tự kiểm |
| 2 | Xem mục 5 | `sorted(range(n), key=..., reverse=True)` | `ans[i] = rank_to_label(place + 1)` |
| 3 | Như bài 2 nhưng không đổi sang chuỗi | `ranks = [0] * n` | `ranks[i] = place + 1` |
| 4 | Duyệt `order`, nhớ điểm trước đó | nếu điểm bằng điểm trước → giữ hạng cũ | ngược lại hạng = `place + 1` |
| 5 | argsort lấy 3 người đầu | `order[:3]` | `{MEDALS[p]: names[i] for p, i in enumerate(order[:3])}` |
| 6 | Key là tuple | `key=lambda i: (-records[i][0], records[i][1])` | không cần `reverse` |

## 17. Đáp án tham khảo

**Bài 1:** `score = [7, 30, 2, 15, 22, 1]` → `order = [1, 4, 3, 0, 2, 5]` → `answer = ["4", "Gold Medal", "5", "Bronze Medal", "Silver Medal", "6"]`.

In [25]:
print(trace_relative_ranks([7, 30, 2, 15, 22, 1]))

score = [7, 30, 2, 15, 22, 1]
order = [1, 4, 3, 0, 2, 5] (chỉ số theo điểm giảm dần)
place=0 -> i=1, score= 30, hạng=1 -> ans[1]='Gold Medal' | ['_', 'G', '_', '_', '_', '_']
place=1 -> i=4, score= 22, hạng=2 -> ans[4]='Silver Medal' | ['_', 'G', '_', '_', 'S', '_']
place=2 -> i=3, score= 15, hạng=3 -> ans[3]='Bronze Medal' | ['_', 'G', '_', 'B', 'S', '_']
place=3 -> i=0, score=  7, hạng=4 -> ans[0]='4' | ['4', 'G', '_', 'B', 'S', '_']
place=4 -> i=2, score=  2, hạng=5 -> ans[2]='5' | ['4', 'G', '5', 'B', 'S', '_']
place=5 -> i=5, score=  1, hạng=6 -> ans[5]='6' | ['4', 'G', '5', 'B', 'S', '6']
['4', 'Gold Medal', '5', 'Bronze Medal', 'Silver Medal', '6']


In [26]:
def answer_exercise_relative_ranks(score):
    n = len(score)
    order = sorted(range(n), key=lambda i: score[i], reverse=True)
    ans = [""] * n
    for place, i in enumerate(order):
        ans[i] = rank_to_label(place + 1)
    return ans

def answer_numeric_ranks(score):
    order = sorted(range(len(score)), key=lambda i: score[i], reverse=True)
    ranks = [0] * len(score)
    for place, i in enumerate(order):
        ranks[i] = place + 1
    return ranks

def answer_competition_ranks(score):
    order = sorted(range(len(score)), key=lambda i: score[i], reverse=True)
    ranks = [0] * len(score)
    for place, i in enumerate(order):
        if place > 0 and score[i] == score[order[place - 1]]:
            ranks[i] = ranks[order[place - 1]]
        else:
            ranks[i] = place + 1
    return ranks

def answer_medal_winners(names, score):
    order = sorted(range(len(score)), key=lambda i: score[i], reverse=True)
    return {MEDALS[p]: names[i] for p, i in enumerate(order[:3])}

def answer_multi_key_ranks(records):
    order = sorted(range(len(records)), key=lambda i: (-records[i][0], records[i][1]))
    ranks = [0] * len(records)
    for place, i in enumerate(order):
        ranks[i] = place + 1
    return ranks

assert answer_exercise_relative_ranks([10, 3, 8, 9, 4]) == ["Gold Medal", "5", "Bronze Medal", "Silver Medal", "4"]
assert answer_numeric_ranks([10, 3, 8, 9, 4]) == [1, 5, 3, 2, 4]
assert answer_competition_ranks([90, 80, 80, 70]) == [1, 2, 2, 4]
assert answer_medal_winners(["An", "Bình", "Chi", "Dũng"], [5, 9, 7, 1]) == {"Gold Medal": "Bình", "Silver Medal": "Chi", "Bronze Medal": "An"}
assert answer_medal_winners(["An"], [3]) == {"Gold Medal": "An"}
assert answer_multi_key_ranks([(90, 30), (95, 50), (90, 20)]) == [3, 1, 2]
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
order = sorted(range(n), key=lambda i: score[i], reverse=True)   # argsort giảm dần
ans = [""] * n
for place, i in enumerate(order):                                  # place: 0-based
    ans[i] = medals[place] if place < 3 else str(place + 1)        # ghi về vị trí GỐC
```

Câu nhớ: **"Sắp chỉ số, đọc hạng theo vị trí, ghi về chỗ cũ."**

### Checklist
- [ ] Sắp **giảm dần** (điểm cao = hạng 1).
- [ ] Ghi vào `ans[i]` (chỉ số gốc), không phải `ans[place]`.
- [ ] Hạng = `place + 1`; 3 hạng đầu là chuỗi huy chương **đúng chính tả**.
- [ ] Hạng ≥ 4 là **chuỗi** `str(...)`, không phải số.
- [ ] Time `O(n log n)`, Space `O(n)`.